# TAXSIM35 Bridge — Marriage Subsidy Calculation


### One-time Terminal setup
```bash
curl -o ~/taxsim35 https://taxsim.nber.org/stata/taxsim35/taxsim35-osx.exe
chmod +x ~/taxsim35
printf 'taxsimid,mstat,year,pwages\n1,1,2012,50000' | ~/taxsim35
```

In [10]:
import io
import os
import subprocess
import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────
LASSO_PKL  = "/Users/trinityrose/Desktop/Replication Project/acs_ssc_predicted_second.pkl"
OUTPUT_PKL = "/Users/trinityrose/Desktop/Replication Project/acs_ssc_taxsim_results_second.pkl"
TAXSIM_EXE = os.path.expanduser("~/taxsim35")

if not os.path.exists(TAXSIM_EXE):
    raise FileNotFoundError(
        f"Binary not found at {TAXSIM_EXE}.\n"
        "Run: curl -o ~/taxsim35 "
        "https://taxsim.nber.org/stata/taxsim35/taxsim35-osx.exe "
        "&& chmod +x ~/taxsim35"
    )

df = pd.read_pickle(LASSO_PKL)
print(f"Loaded: {df.shape}")
print(f"Years:  {sorted(df['year'].unique())}")
print(f"Married:    {(df['sscouple_mar']==True).sum():,}")
print(f"Cohabiting: {(df['sscouple_mar']==False).sum():,}")

Loaded: (37907, 72)
Years:  [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
Married:    16,146
Cohabiting: 21,761


In [11]:
# ── FIPS → SOI state code conversion ─────────────────────────────
# TAXSIM uses IRS SOI codes (sequential 1-51), not Census FIPS.

FIPS_TO_SOI = {
    1:  1,  2:  2,  4:  3,  5:  4,  6:  5,  8:  6,  9:  7,
    10: 8,  11: 9,  12: 10, 13: 11, 15: 12, 16: 13, 17: 14,
    18: 15, 19: 16, 20: 17, 21: 18, 22: 19, 23: 20, 24: 21,
    25: 22, 26: 23, 27: 24, 28: 25, 29: 26, 30: 27, 31: 28,
    32: 29, 33: 30, 34: 31, 35: 32, 36: 33, 37: 34, 38: 35,
    39: 36, 40: 37, 41: 38, 42: 39, 44: 40, 45: 41, 46: 42,
    47: 43, 48: 44, 49: 45, 50: 46, 51: 47, 53: 48, 54: 49,
    55: 50, 56: 51,
}

unmapped = set(df["statefip"].unique()) - set(FIPS_TO_SOI.keys())
print(f"Unmapped FIPS: {unmapped if unmapped else 'None'}")

Unmapped FIPS: None


In [12]:
# ── TAXSIM35 chunked runner ───────────────────────────────────────

def run_taxsim35(df_taxsim: pd.DataFrame, chunksize: int = 1000,
                 label: str = "") -> pd.DataFrame:
    chunks_out = []
    n = len(df_taxsim)
    n_chunks = (n + chunksize - 1) // chunksize
    for i in range(n_chunks):
        chunk = df_taxsim.iloc[i * chunksize : (i + 1) * chunksize].copy()
        chunk["taxsimid"] = np.arange(1, len(chunk) + 1)
        result = subprocess.run(
            [TAXSIM_EXE],
            input=chunk.to_csv(index=False),
            capture_output=True, text=True, timeout=300,
        )
        stdout = result.stdout.strip()
        if not stdout or not stdout.startswith("taxsimid"):
            raise RuntimeError(
                f"{label} chunk {i+1}/{n_chunks} failed.\n"
                f"stderr: {result.stderr[:1000]}\n"
                f"stdout: {stdout[:500]}"
            )
        chunks_out.append(pd.read_csv(io.StringIO(result.stdout)))
        if (i + 1) % 10 == 0 or i == n_chunks - 1:
            print(f"  {label}: {i+1}/{n_chunks} chunks "
                  f"({min((i+1)*chunksize, n):,}/{n:,} rows)")
    return pd.concat(chunks_out, ignore_index=True)


# ── Input builders ────────────────────────────────────────────────
# Per paper p.89: 'no other tax expenditures' means depx=0 in ALL runs.
# Children are controlled for separately in the regression.
# Both single filers use mstat=1 (plain single, no HoH).

def make_single(df, wage_col, age_col):
    """Single filer — wages only, no children, plain mstat=1."""
    t = pd.DataFrame()
    t["taxsimid"] = np.arange(1, len(df) + 1)
    t["year"]   = df["year"].values
    t["state"]  = df["statefip"].map(FIPS_TO_SOI)
    t["mstat"]  = 1
    t["page"]   = df[age_col].values
    t["sage"]   = 0
    t["pwages"] = df[wage_col].clip(lower=0).fillna(0).round().astype(int)
    t["swages"] = 0
    t["depx"]   = 0  
    return t


def make_joint(df, wage_r_col, wage_sp_col, age_r_col, age_sp_col):
    """Joint filer — wages only, no children, mstat=2."""
    t = pd.DataFrame()
    t["taxsimid"] = np.arange(1, len(df) + 1)
    t["year"]   = df["year"].values
    t["state"]  = df["statefip"].map(FIPS_TO_SOI)
    t["mstat"]  = 2
    t["page"]   = df[age_r_col].values
    t["sage"]   = df[age_sp_col].values
    t["pwages"] = df[wage_r_col].clip(lower=0).fillna(0).round().astype(int)
    t["swages"] = df[wage_sp_col].clip(lower=0).fillna(0).round().astype(int)
    t["depx"]   = 0   
    return t


print("Helpers defined.")

Helpers defined.


In [13]:
# ══════════════════════════════════════════════════════════════════
# RUN A — PREDICTED INCOME  (instrument for Table 3 IV)
# Uses hat_incearn_r / hat_incearn_sp from LASSO
# ══════════════════════════════════════════════════════════════════

print("Run A1: respondent as single (predicted)...")
a1_out = run_taxsim35(
    make_single(df, "hat_incearn_r", "age"),
    label="A1")

print("Run A2: spouse as single (predicted)...")
a2_out = run_taxsim35(
    make_single(df, "hat_incearn_sp", "sp_age"),
    label="A2")

print("Run A3: couple jointly (predicted)...")
a3_out = run_taxsim35(
    make_joint(df, "hat_incearn_r", "hat_incearn_sp", "age", "sp_age"),
    label="A3")

assert len(a1_out) == len(a2_out) == len(a3_out) == len(df), "Row mismatch!"
print(f"\nA runs done.")
print(f"  A1 mean fiitax: {a1_out['fiitax'].mean():,.2f}")
print(f"  A2 mean fiitax: {a2_out['fiitax'].mean():,.2f}")
print(f"  A3 mean fiitax: {a3_out['fiitax'].mean():,.2f}")
print(f"  Raw fed subsidy (all): {(a1_out['fiitax']+a2_out['fiitax']-a3_out['fiitax']).mean():,.2f}")

Run A1: respondent as single (predicted)...
  A1: 10/38 chunks (10,000/37,907 rows)
  A1: 20/38 chunks (20,000/37,907 rows)
  A1: 30/38 chunks (30,000/37,907 rows)
  A1: 38/38 chunks (37,907/37,907 rows)
Run A2: spouse as single (predicted)...
  A2: 10/38 chunks (10,000/37,907 rows)
  A2: 20/38 chunks (20,000/37,907 rows)
  A2: 30/38 chunks (30,000/37,907 rows)
  A2: 38/38 chunks (37,907/37,907 rows)
Run A3: couple jointly (predicted)...
  A3: 10/38 chunks (10,000/37,907 rows)
  A3: 20/38 chunks (20,000/37,907 rows)
  A3: 30/38 chunks (30,000/37,907 rows)
  A3: 38/38 chunks (37,907/37,907 rows)

A runs done.
  A1 mean fiitax: 9,913.02
  A2 mean fiitax: 6,129.85
  A3 mean fiitax: 15,817.73
  Raw fed subsidy (all): 225.13


In [14]:
print([c for c in df.columns if 'hat' in c or 'incearn' in c or 'lasso' in c.lower()])

['r_incearn', 'sp_incearn', 'c_incearn', 'c_incearn_split', 'c_incearn1', 'c_incearn_split1', 'c_incearn2', 'c_incearn_split2', 'c_incearn3', 'c_incearn_split3', 'c_incearn4', 'c_incearn_split4', 'c_incearn5', 'c_incearn_split5', 'r_posincearn', 'sp_posincearn', 'hat_pos_r', 'hat_incearn_r', 'hat_pos_sp', 'hat_incearn_sp', 'hat_c_incearn', 'hat_c_split', 'hat_c_incearn1', 'hat_c_split1', 'hat_c_incearn2', 'hat_c_split2', 'hat_c_incearn3', 'hat_c_split3', 'hat_c_incearn4', 'hat_c_split4', 'hat_c_incearn5', 'hat_c_split5']


In [15]:
# ══════════════════════════════════════════════════════════════════
# RUN B — REPORTED INCOME  (observed subsidy for Table 2 + OLS)
# Uses r_incearn / sp_incearn directly from ACS
# ══════════════════════════════════════════════════════════════════

print("Run B1: respondent as single (reported)...")
b1_out = run_taxsim35(
    make_single(df, "r_incearn", "age"),
    label="B1")

print("Run B2: spouse as single (reported)...")
b2_out = run_taxsim35(
    make_single(df, "sp_incearn", "sp_age"),
    label="B2")

print("Run B3: couple jointly (reported)...")
b3_out = run_taxsim35(
    make_joint(df, "r_incearn", "sp_incearn", "age", "sp_age"),
    label="B3")

assert len(b1_out) == len(b2_out) == len(b3_out) == len(df), "Row mismatch!"
print(f"\nB runs done.")
print(f"  Raw fed subsidy (all): {(b1_out['fiitax']+b2_out['fiitax']-b3_out['fiitax']).mean():,.2f}")

Run B1: respondent as single (reported)...
  B1: 10/38 chunks (10,000/37,907 rows)
  B1: 20/38 chunks (20,000/37,907 rows)
  B1: 30/38 chunks (30,000/37,907 rows)
  B1: 38/38 chunks (37,907/37,907 rows)
Run B2: spouse as single (reported)...
  B2: 10/38 chunks (10,000/37,907 rows)
  B2: 20/38 chunks (20,000/37,907 rows)
  B2: 30/38 chunks (30,000/37,907 rows)
  B2: 38/38 chunks (37,907/37,907 rows)
Run B3: couple jointly (reported)...
  B3: 10/38 chunks (10,000/37,907 rows)
  B3: 20/38 chunks (20,000/37,907 rows)
  B3: 30/38 chunks (30,000/37,907 rows)
  B3: 38/38 chunks (37,907/37,907 rows)

B runs done.
  Raw fed subsidy (all): 686.62


In [16]:
# ── Compute marriage subsidies ────────────────────────────────────
# Subsidy = (T_i + T_j) - T_c
# Positive = subsidy (married pay less). Negative = penalty.

df_out = df.copy()
m = df_out["sscouple_mar"] == True
c = df_out["sscouple_mar"] == False

# Predicted income subsidies (instrument)
df_out["fed_sub_pred"]   = (a1_out["fiitax"].values + a2_out["fiitax"].values
                            - a3_out["fiitax"].values)
df_out["state_sub_pred"] = (a1_out["siitax"].values + a2_out["siitax"].values
                            - a3_out["siitax"].values)
df_out["total_sub_pred"] = df_out["fed_sub_pred"] + df_out["state_sub_pred"]

# Reported income subsidies (observed)
df_out["fed_sub_obs"]    = (b1_out["fiitax"].values + b2_out["fiitax"].values
                            - b3_out["fiitax"].values)
df_out["state_sub_obs"]  = (b1_out["siitax"].values + b2_out["siitax"].values
                            - b3_out["siitax"].values)
df_out["total_sub_obs"]  = df_out["fed_sub_obs"] + df_out["state_sub_obs"]

# Joint filing totals (for other analyses)
df_out["fiitax_joint_pred"] = a3_out["fiitax"].values
df_out["siitax_joint_pred"] = a3_out["siitax"].values
df_out["fica_joint_pred"]   = a3_out["fica"].values
df_out["fiitax_joint_obs"]  = b3_out["fiitax"].values
df_out["siitax_joint_obs"]  = b3_out["siitax"].values
df_out["fica_joint_obs"]    = b3_out["fica"].values

print("Subsidies computed.")
print(f"\nQuick check vs paper:")
print(f"  Fed sub pred  married: {df_out.loc[m,'fed_sub_pred'].mean():>10,.2f}  (paper: 122.41)")
print(f"  Fed sub pred  cohab:   {df_out.loc[c,'fed_sub_pred'].mean():>10,.2f}  (paper: 266.89)")
print(f"  Fed sub obs   married: {df_out.loc[m,'fed_sub_obs'].mean():>10,.2f}  (paper: 395.05)")
print(f"  Fed sub obs   cohab:   {df_out.loc[c,'fed_sub_obs'].mean():>10,.2f}  (paper: 231.80)")
print(f"  State sub pred married:{df_out.loc[m,'state_sub_pred'].mean():>10,.2f}  (paper: -54.21)")
print(f"  State sub pred cohab:  {df_out.loc[c,'state_sub_pred'].mean():>10,.2f}  (paper: -10.06)")

Subsidies computed.

Quick check vs paper:
  Fed sub pred  married:     253.92  (paper: 122.41)
  Fed sub pred  cohab:       203.77  (paper: 266.89)
  Fed sub obs   married:     879.69  (paper: 395.05)
  Fed sub obs   cohab:       543.36  (paper: 231.80)
  State sub pred married:    -29.93  (paper: -54.21)
  State sub pred cohab:      -38.04  (paper: -10.06)


In [17]:
# ── Replicate Table 2 ─────────────────────────────────────────────

def ms(series, mask, fmt=".2f"):
    v = series[mask]
    return f"{v.mean():{fmt}}  ({v.std():{fmt}})"

rep_total    = df_out["r_incearn"] + df_out["sp_incearn"]
pos_rep_m    = m & (rep_total > 0)
pos_rep_c    = c & (rep_total > 0)
rep_split_m  = (df_out.loc[pos_rep_m, ["r_incearn","sp_incearn"]].max(axis=1)
                / rep_total[pos_rep_m])
rep_split_c  = (df_out.loc[pos_rep_c, ["r_incearn","sp_incearn"]].max(axis=1)
                / rep_total[pos_rep_c])
pos_pred_m   = m & (df_out["hat_c_incearn"] > 0)
pos_pred_c   = c & (df_out["hat_c_incearn"] > 0)
pred_split_m = df_out.loc[pos_pred_m, "hat_c_split"]
pred_split_c = df_out.loc[pos_pred_c, "hat_c_split"]

rows = [
    ("Positive earnings (reported)",
     ms((rep_total>0).astype(float), m, ".3f"),   "0.935 (0.246)",
     ms((rep_total>0).astype(float), c, ".3f"),   "0.935 (0.247)"),
    ("Positive earnings (predicted)",
     ms((df_out["hat_incearn_r"]+df_out["hat_incearn_sp"]>0).astype(float), m, ".3f"),
     "0.963 (0.188)",
     ms((df_out["hat_incearn_r"]+df_out["hat_incearn_sp"]>0).astype(float), c, ".3f"),
     "0.969 (0.172)"),
    ("Reported earnings",
     ms(rep_total, m, ",.2f"),   "125,286.76 (119,779.91)",
     ms(rep_total, c, ",.2f"),   "105,188.00 (105,191.59)"),
    ("Predicted earnings",
     ms(df_out["hat_c_incearn"], m, ",.2f"),   "110,729.40 (57,936.40)",
     ms(df_out["hat_c_incearn"], c, ",.2f"),   "102,952.54 (54,275.74)"),
    ("Reported earnings split",
     f"{rep_split_m.mean():.3f}  ({rep_split_m.std():.3f})",   "0.745 (0.200)",
     f"{rep_split_c.mean():.3f}  ({rep_split_c.std():.3f})",   "0.723 (0.174)"),
    ("Predicted earnings split",
     f"{pred_split_m.mean():.3f}  ({pred_split_m.std():.3f})",   "0.648 (0.197)",
     f"{pred_split_c.mean():.3f}  ({pred_split_c.std():.3f})",   "0.641 (0.181)"),
    ("Fed+state subsidy (reported)",
     ms(df_out["total_sub_obs"], m, ",.2f"),   "442.45 (5,116.62)",
     ms(df_out["total_sub_obs"], c, ",.2f"),   "263.79 (3,247.05)"),
    ("Fed+state subsidy (predicted)",
     ms(df_out["total_sub_pred"], m, ",.2f"),   "68.19 (2,218.99)",
     ms(df_out["total_sub_pred"], c, ",.2f"),   "256.82 (1,623.22)"),
    ("Fed subsidy (reported)",
     ms(df_out["fed_sub_obs"], m, ",.2f"),   "395.05 (4,563.36)",
     ms(df_out["fed_sub_obs"], c, ",.2f"),   "231.80 (3,055.28)"),
    ("Fed subsidy (predicted)",
     ms(df_out["fed_sub_pred"], m, ",.2f"),   "122.41 (1,896.07)",
     ms(df_out["fed_sub_pred"], c, ",.2f"),   "266.89 (1,427.33)"),
    ("State subsidy (reported)",
     ms(df_out["state_sub_obs"], m, ",.2f"),   "47.41 (974.14)",
     ms(df_out["state_sub_obs"], c, ",.2f"),   "31.99 (584.34)"),
    ("State subsidy (predicted)",
     ms(df_out["state_sub_pred"], m, ",.2f"),   "-54.21 (487.06)",
     ms(df_out["state_sub_pred"], c, ",.2f"),   "-10.06 (332.98)"),
]

w = 42
print(f"{'TABLE 2':^126}")
print(f"{'':>{w}} {'Replication (Married)':>28} {'Paper (Married)':>24} "
      f"{'Replication (Cohab)':>28} {'Paper (Cohab)':>22}")
print("─" * 126)
for label, ym, pm, yc, pc in rows:
    print(f"{label:<{w}} {ym:>28} {pm:>24} {yc:>28} {pc:>22}")
print("─" * 126)
print(f"{'Observations':<{w}} {m.sum():>28,} {'16,098':>24} "
      f"{c.sum():>28,} {'21,136':>22}")

                                                           TABLE 2                                                            
                                                  Replication (Married)          Paper (Married)          Replication (Cohab)          Paper (Cohab)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Positive earnings (reported)                             0.971  (0.168)            0.935 (0.246)               0.971  (0.167)          0.935 (0.247)
Positive earnings (predicted)                            0.990  (0.099)            0.963 (0.188)               0.994  (0.077)          0.969 (0.172)
Reported earnings                              125,125.11  (119,751.06)  125,286.76 (119,779.91)     104,319.22  (104,848.01) 105,188.00 (105,191.59)
Predicted earnings                              115,730.02  (49,401.20)   110,729.40 (57,936.40)      108,428.94  (46,770.59) 102,952.54 (54,275.74)


In [18]:
# ── Subsidy by year ───────────────────────────────────────────────
print("=== Mean predicted total subsidy by year ===")
piv = df_out.groupby(["year","sscouple_mar"])["total_sub_pred"].mean().unstack()
piv.columns = ["Cohabiting", "Married"]
piv["Married - Cohabiting"] = piv["Married"] - piv["Cohabiting"]
print(piv.round(1).to_string())

print("\n=== Mean observed total subsidy by year ===")
piv2 = df_out.groupby(["year","sscouple_mar"])["total_sub_obs"].mean().unstack()
piv2.columns = ["Cohabiting", "Married"]
piv2["Married - Cohabiting"] = piv2["Married"] - piv2["Cohabiting"]
print(piv2.round(1).to_string())

=== Mean predicted total subsidy by year ===
      Cohabiting  Married  Married - Cohabiting
year                                           
2012       134.9    231.1                  96.2
2013       134.3    275.7                 141.4
2014       189.8    233.0                  43.3
2015       173.8    225.0                  51.3
2016       186.5    224.6                  38.0
2017       182.9    192.3                   9.4

=== Mean observed total subsidy by year ===
      Cohabiting  Married  Married - Cohabiting
year                                           
2012       761.6   1129.2                 367.6
2013       728.9   1081.1                 352.1
2014       632.7   1034.0                 401.3
2015       620.3   1228.8                 608.5
2016       524.5    999.6                 475.1
2017       448.0    914.7                 466.7


In [19]:
# ── Save ──────────────────────────────────────────────────────────
new_cols = [
    "fed_sub_pred",  "state_sub_pred", "total_sub_pred",
    "fed_sub_obs",   "state_sub_obs",  "total_sub_obs",
    "fiitax_joint_pred", "siitax_joint_pred", "fica_joint_pred",
    "fiitax_joint_obs",  "siitax_joint_obs",  "fica_joint_obs",
]
df_out.to_pickle(OUTPUT_PKL)
print(f"Saved → {OUTPUT_PKL}")
print(f"Shape: {df_out.shape}")
print(f"New columns: {new_cols}")

Saved → /Users/trinityrose/Desktop/Replication Project/acs_ssc_taxsim_results_second.pkl
Shape: (37907, 84)
New columns: ['fed_sub_pred', 'state_sub_pred', 'total_sub_pred', 'fed_sub_obs', 'state_sub_obs', 'total_sub_obs', 'fiitax_joint_pred', 'siitax_joint_pred', 'fica_joint_pred', 'fiitax_joint_obs', 'siitax_joint_obs', 'fica_joint_obs']
